In [1]:
import os
import io
import zipfile
import requests
import frontmatter
import re
import numpy as np
import warnings
from typing import List, Any
from tqdm.auto import tqdm
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_agent
warnings.filterwarnings("ignore", message="Core Pydantic V1 functionality")

os.environ["OPENROUTER_API_KEY"]

c:\Users\rabi3\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\rabi3\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
def ingest_fastapi_docs():
    url = 'https://codeload.github.com/fastapi/fastapi/zip/refs/heads/master'
    resp = requests.get(url)
    if resp.status_code != 200:
        raise Exception("Failed to download repository")

    repo_data = []
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for file_info in zf.infolist():
            filename = file_info.filename
            if 'docs/en/docs/' in filename and filename.endswith('.md'):
                with zf.open(file_info) as f_in:
                    content = f_in.read().decode('utf-8', errors='ignore')
                    post = frontmatter.loads(content)
                    data = post.to_dict()
                    data['content'] = post.content
                    data['filename'] = filename
                    repo_data.append(data)
    return repo_data

fastapi_data = ingest_fastapi_docs()
print(f"Successfully ingested {len(fastapi_data)} documentation files!")

if fastapi_data:
    print("\nSample Data Keys:", fastapi_data[0].keys())
    print("Sample Filename:", fastapi_data[0]['filename'])

Successfully ingested 153 documentation files!

Sample Data Keys: dict_keys(['content', 'filename'])
Sample Filename: fastapi-master/docs/en/docs/_llm-test.md


In [3]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive") 
    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk}) 
        if i + size >= n:
            break
    return result

fastapi_sliding_chunks = []
for doc in fastapi_data:
    doc_copy = doc.copy()
    content = doc_copy.pop('content')
    chunks = sliding_window(content, 2000, 1500) 
    for chunk in chunks:
        chunk.update(doc_copy)
        fastapi_sliding_chunks.append(chunk)

print(f"Created {len(fastapi_sliding_chunks)} sliding window chunks.")

Created 1013 sliding window chunks.


In [4]:
def split_markdown_by_level(text, level=2):
    header_pattern = r'^(\#{' + str(level) + r'})(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)
    parts = pattern.split(text)
    sections = []
    for i in range(1, len(parts), 3):
        header = (parts[i] + parts[i+1]).strip()
        content = parts[i+2].strip() if i+2 < len(parts) else ""
        if content:
            sections.append(f"{header}\n\n{content}")
    return sections

fastapi_chunks = []
for doc in fastapi_data:
    sections = split_markdown_by_level(doc['content'], level=2)
    doc_title = doc.get('title', doc.get('filename', 'FastAPI Doc'))
    for section in sections:
        fastapi_chunks.append({
            'title': doc_title,
            'filename': doc['filename'],
            'section_content': section
        })

print(f"Created {len(fastapi_chunks)} section-based chunks.")

Created 1729 section-based chunks.


In [5]:
# Lexical Index
text_index = Index(
    text_fields=["section_content", "title", "filename"],
    keyword_fields=[]
)

text_index.fit(fastapi_chunks)

query = "How do I handle OAuth2 security?"
text_results = text_index.search(query, num_results=3)
print("Top Text Search Result:", text_results[0]['title'])

Top Text Search Result: fastapi-master/docs/en/docs/advanced/security/oauth2-scopes.md


In [6]:
# Vector Index
model = SentenceTransformer('multi-qa-distilbert-cos-v1')

embeddings = []
for d in tqdm(fastapi_chunks):
    v = model.encode(d['section_content'])
    embeddings.append(v)

vector_index = VectorSearch()
vector_index.fit(np.array(embeddings), fastapi_chunks)

v_query = model.encode("How can I secure my API?")
vector_results = vector_index.search(v_query, num_results=3)

100%|██████████| 1729/1729 [00:19<00:00, 86.83it/s] 


In [7]:
def hybrid_search(query: str, num_results=5) -> List[Any]:
    t_results = text_index.search(query, num_results=num_results)
    v_query = model.encode(query)
    v_results = vector_index.search(v_query, num_results=num_results)
    
    seen = set()
    combined = []
    for res in t_results + v_results:
        uid = res['filename'] + res['section_content'][:50]
        if uid not in seen:
            seen.add(uid)
            combined.append(res)
    return combined[:num_results]

test_query = "getting started with OAuth2 password flow"

results = hybrid_search(test_query, num_results=3)

print(f"--- Hybrid Search Results for: '{test_query}' ---")
for i, res in enumerate(results):
    print(f"\nResult {i+1}: {res['title']}")
    print(f"Source: {res['filename']}")
    print(f"Preview: {res['section_content'][:150]}...")

--- Hybrid Search Results for: 'getting started with OAuth2 password flow' ---

Result 1: fastapi-master/docs/en/docs/advanced/security/oauth2-scopes.md
Source: fastapi-master/docs/en/docs/advanced/security/oauth2-scopes.md
Preview: ## About third party integrations { #about-third-party-integrations }

In this example we are using the OAuth2 "password" flow.

This is appropriate w...

Result 2: fastapi-master/docs/en/docs/tutorial/security/simple-oauth2.md
Source: fastapi-master/docs/en/docs/tutorial/security/simple-oauth2.md
Preview: ## Get the `username` and `password` { #get-the-username-and-password }

We are going to use **FastAPI** security utilities to get the `username` and ...

Result 3: fastapi-master/docs/en/docs/tutorial/security/simple-oauth2.md
Source: fastapi-master/docs/en/docs/tutorial/security/simple-oauth2.md
Preview: #### Password hashing { #password-hashing }

"Hashing" means: converting some content (a password in this case) into a sequence of bytes (just a strin.

In [8]:
@tool
def search_fastapi_docs(query: str):
    """
    Search the FastAPI documentation for technical details and examples.
    Use this for any questions about FastAPI implementation, security, or syntax.
    """
    
    return hybrid_search(query)

tools = [search_fastapi_docs]

In [9]:
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.0,          
)

system_prompt = """
You are a FastAPI expert. 
Use the 'search_fastapi_docs' tool to find official documentation before answering any technical question.
Always provide accurate code examples when they are available in the retrieved context.
Be clear, concise and include import statements when showing code.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)

In [10]:
question = "How do I implement OAuth2 with a password flow in FastAPI?"

response = agent.invoke({
    "messages": [("human", question)]
})

final_output = response["messages"][-1].content
print(final_output)

To implement OAuth2 with the password flow (Resource Owner Password Credentials Grant) in FastAPI, you'll need to use a third-party library since FastAPI core doesn't include native OAuth2 support. The most straightforward and recommended approach is to use the [`fastapi-users`](https://fastapi-users.github.io/fastapi-users/) library, which provides built-in support for OAuth2 flows including the password grant.

### Why `fastapi-users`?
- Handles user management, authentication, and OAuth2 integration seamlessly with FastAPI.
- Supports password flow via its OAuth2 router (`/oauth2/token` endpoint).
- Uses secure JWT handling under the hood (with `python-jose`).
- Avoids reinventing complex auth logic (token generation, validation, etc.).

### Important Security Note
The OAuth2 password flow **is not recommended for public-facing applications** because it requires clients to handle user credentials directly (username/password), increasing risks of credential theft. For public apps, pr